# Level-3.5 Neuro-Symbolic Intent System

This notebook exercises the full Level-3.5 pipeline: semantic parsing → ontology-grounded frame construction → symbolic reasoning → plan generation → response synthesis.

In [54]:
# Cell 1 — Imports & Repo Setup

import os
import sys
import importlib
import pandas as pd

print("✓ Basic imports complete")


✓ Basic imports complete


In [55]:
# Cell 2 — Locate repo root and add to sys.path

def find_repo_root(start_dir=None):
    d = start_dir or os.getcwd()
    while True:
        if os.path.exists(os.path.join(d, '.git')) or os.path.exists(os.path.join(d, 'requirements.txt')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.getcwd()
        d = parent

repo_root = find_repo_root()
sys.path.insert(0, repo_root)

print(f"✓ Repo root detected: {repo_root}")

✓ Repo root detected: c:\git\nsai_poc


In [56]:
# Cell 3 — Import Level-3.5 components (force reload from disk)

# Purge any stale/broken level3_5 module entries from sys.modules so
# Python re-imports every .py file from disk on the next import.
# This is safer than importlib.reload when a prior import failed
# and left a partial module object in sys.modules.
for _mod_key in list(sys.modules.keys()):
    if _mod_key == \"level3_5\" or _mod_key.startswith("level3_5."):
        del sys.modules[_mod_key]

# Fresh imports in dependency order
import level3_5.ontology
import level3_5.semantic_parser
import level3_5.reasoner
import level3_5.planner
import level3_5.responder
import level3_5.pipeline
import level3_5.intent_model

from level3_5.ontology import INTENTS, ENTITIES
from level3_5.semantic_parser import parse_utterance
from level3_5.reasoner import reason
from level3_5.planner import create_plan
from level3_5.responder import generate_response
from level3_5.pipeline import run_pipeline

print("✓ Level-3.5 modules reloaded")


✓ Level-3.5 modules reloaded


In [57]:
# Cell 4 — Load base intent dataset

data_path = os.path.join(repo_root, "data", "intents_base.csv")

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}")

df = pd.read_csv(data_path)

print(f"✓ Loaded {len(df)} records")
print("\nSample rows:")
display(df.head())

✓ Loaded 1661 records

Sample rows:


,utterance,intent
0,summarize the horizontal pod autoscaler activity,summarization
1,provide a recap of the cost trends by environment,summarization
2,how do I play basketball,out_of_scope
3,configure the health check interval to 30 seconds,execution
4,inject a latency fault into the payment service,execution


In [58]:
# Train TF-IDF Intent Model (force reload from disk)

# Purge any stale/broken level3_5 module entries from sys.modules so
# Python re-imports every .py file from disk on the next import.
for _mod_key in list(sys.modules.keys()):
    if _mod_key == \"level3_5\" or _mod_key.startswith("level3_5."):
        del sys.modules[_mod_key]

import level3_5.intent_model
from level3_5.intent_model import IntentClassifier

classifier = IntentClassifier()
classifier.train(df)

print("✓ Intent model trained and saved.")


✓ Intent model trained and saved.


In [59]:
# Cell 5 — Dataset validation

required_cols = ["utterance", "intent"]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("✓ Dataset contains required columns")
print("\nIntent distribution:")
print(df["intent"].value_counts())

✓ Dataset contains required columns

Intent distribution:
intent
out_of_scope     480
execution        413
investigate      412
summarization    356
Name: count, dtype: int64


In [60]:
# Cell 6 — Run a single example through pipeline

example = df.iloc[0]["utterance"]

result = run_pipeline(example)

print("Utterance:", example)
print("\nStructured Frame:")
print(result["frame"])

print("\nReasoning Output:")
print(result["reasoning"])

print("\nPlan:")
print(result["plan"])

print("\nFinal Response:")
print(result["response"])

Utterance: summarize the horizontal pod autoscaler activity

Structured Frame:
{'intent': 'summarization', 'entity': 'horizontal_pod_autoscaler', 'symptom': None, 'time_context': None, 'confidence': {'intent': 0.7976457693199407}}

Reasoning Output:
{'mode': 'reporting', 'target': 'horizontal_pod_autoscaler', 'report_type': 'summary'}

Plan:
{'steps': ['fetch_recent_data', 'aggregate', 'generate_summary']}

Final Response:
Intent: summarization
Entity: horizontal_pod_autoscaler
Symptom: None
Time Context: None
Planned Steps: ['fetch_recent_data', 'aggregate', 'generate_summary']


In [61]:
# Cell 7 — Apply Level-3.5 pipeline to full dataset

outputs = []

for _, row in df.iterrows():
    utterance = row["utterance"]
    gold_intent = row["intent"]

    result = run_pipeline(utterance)

    outputs.append({
        "utterance": utterance,
        "gold_intent": gold_intent,
        "parsed_intent": result["frame"]["intent"],
        "entity": result["frame"]["entity"],
        "symptom": result["frame"]["symptom"],
        "time_context": result["frame"]["time_context"],
        "reasoning_mode": result["reasoning"].get("mode")
    })

level3_5_df = pd.DataFrame(outputs)

print("✓ Level-3.5 processing complete")
display(level3_5_df.head())

✓ Level-3.5 processing complete


,utterance,gold_intent,parsed_intent,entity,symptom,time_context,reasoning_mode
0,summarize the horizontal pod autoscaler activity,summarization,summarization,horizontal_pod_autoscaler,NaN,NaN,reporting
1,provide a recap of the cost trends by environment,summarization,summarization,cost_trends,NaN,NaN,reporting
2,how do I play basketball,out_of_scope,out_of_scope,NaN,NaN,NaN,reject
3,configure the health check interval to 30 seconds,execution,execution,health_check,NaN,NaN,action
4,inject a latency fault into the payment service,execution,execution,payment_service,latency_high,NaN,action


In [62]:
# Cell 8 — Intent alignment check

comparison = level3_5_df.copy()

comparison["intent_match"] = (
    comparison["gold_intent"].str.lower() ==
    comparison["parsed_intent"].str.lower()
)

accuracy = comparison["intent_match"].mean()

print(f"Intent Match Rate (Parser vs Gold): {accuracy:.2%}")
display(comparison.head())

Intent Match Rate (Parser vs Gold): 99.88%


,utterance,gold_intent,parsed_intent,entity,symptom,time_context,reasoning_mode,intent_match
0,summarize the horizontal pod autoscaler activity,summarization,summarization,horizontal_pod_autoscaler,NaN,NaN,reporting,True
1,provide a recap of the cost trends by environment,summarization,summarization,cost_trends,NaN,NaN,reporting,True
2,how do I play basketball,out_of_scope,out_of_scope,NaN,NaN,NaN,reject,True
3,configure the health check interval to 30 seconds,execution,execution,health_check,NaN,NaN,action,True
4,inject a latency fault into the payment service,execution,execution,payment_service,latency_high,NaN,action,True


In [63]:
# Cell 9 — Entity coverage diagnostics

entity_counts = level3_5_df["entity"].value_counts(dropna=False)

print("Entity distribution:")
print(entity_counts)

Entity distribution:
entity
backup                       448
NaN                          248
kubernetes_node              167
service                       83
database                      53
network_policy                33
health_check                  31
cache                         31
logs                          29
pod                           27
connection_pool               27
message_queue                 26
api                           25
dns                           24
storage                       24
deployment                    23
incident                      21
cost_trends                   19
metrics                       19
alerting                      19
kubernetes_cluster            18
ci_cd_pipeline                18
certificate                   16
load_test                     15
disk_io                       14
reverse_proxy                 13
container_image               12
rate_limiting                 11
resource_quota                11
cost_anomalies 

In [64]:
# Cell 10 — Reasoning mode distribution

mode_counts = level3_5_df["reasoning_mode"].value_counts()

print("Reasoning mode distribution:")
print(mode_counts)

Reasoning mode distribution:
reasoning_mode
reject        479
action        414
diagnostic    412
reporting     356
Name: count, dtype: int64


## Cell 11 — Demonstrate True Level-3.5 Behavior

This cell explicitly demonstrates structured reasoning across in-domain and out-of-domain utterances.

In [65]:
# Cell 11 — Demonstration examples

test_inputs = [
    "why is the backup not completing within the window",
    "restart the message queue consumers",
    "summarize the horizontal pod autoscaler activity",
    "how do I play basketball"
]

for utterance in test_inputs:
    print("="*70)
    result = run_pipeline(utterance)

    print("Utterance:", utterance)
    print("Frame:", result["frame"])
    print("Reasoning:", result["reasoning"])
    print("Plan:", result["plan"])
    print("Response:", result["response"])

Utterance: why is the backup not completing within the window
Frame: {'intent': 'investigate', 'entity': 'backup', 'symptom': 'not_completing', 'time_context': 'sla_window', 'confidence': {'intent': 0.887261153013134}}
Reasoning: {'mode': 'diagnostic', 'target': 'backup', 'entity_type': 'data_protection', 'symptom_focus': 'not_completing', 'time_context': 'sla_window'}
Plan: {'steps': ['collect_metrics', 'collect_logs', 'check_recent_changes', 'analyze_correlations', 'summarize_findings']}
Response: Intent: investigate
Entity: backup
Symptom: not_completing
Time Context: sla_window
Planned Steps: ['collect_metrics', 'collect_logs', 'check_recent_changes', 'analyze_correlations', 'summarize_findings']
Utterance: restart the message queue consumers
Frame: {'intent': 'execution', 'entity': 'message_queue', 'symptom': None, 'time_context': None, 'confidence': {'intent': 0.7961643521213486}}
Reasoning: {'mode': 'action', 'target': 'message_queue', 'requires_confirmation': True}
Plan: {'step

In [66]:
# Cell 12 — Save Level-3.5 structured output

out_path = os.path.join(repo_root, \"level3_5\", "data")
os.makedirs(out_path, exist_ok=True)

save_path = os.path.join(out_path, "level3_5_structured_output.csv")

level3_5_df.to_csv(save_path, index=False)

print(f"✓ Saved Level-3.5 structured dataset to: {save_path}")

✓ Saved Level-3.5 structured dataset to: c:\git\nsai_poc\level3_5\data\level3_5_structured_output.csv


In [ ]:
# ===============================================
# Inference on Unseen / Paraphrased Statements
# ===============================================

unseen_statements = [
    # Paraphrased summarization
    "Give me a summary of cluster health over the last week",
    "Provide an overview of error budget usage this quarter",
    
    # Paraphrased investigate
    "Why are API response times increasing today?",
    "Investigate disk saturation on the database nodes",
    "What is causing consumer lag in Kafka?",
    
    # Execution style
    "Scale up read replicas for production database",
    "Rotate TLS certificates on the API gateway",
    
    # Edge case domain variation
    "Check whether any pods are stuck in pending state",
    "Analyze sudden spike in alert volume",
    
    # Clear out-of-scope
    "Explain how to bake sourdough bread",
    "Who discovered gravity?",
    "How do I learn piano?"
]

print("===== INFERENCE RESULTS =====")

for stmt in unseen_statements:
    result = run_pipeline(stmt)
    
    print("\n" + "="*60)
    print("Utterance:", stmt)
    print("Frame:", result["frame"])
    print("Reasoning:", result["reasoning"])
    print("Plan:", result["plan"])